In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')



In [2]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)

True
2.10.0+cu128


In [3]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(dataset)
print("\nSample text:")
print(dataset['train'][1]['text'])


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Sample text:
 = Valkyria Chronicles III = 



In [4]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Test it
sample = "The quick brown fox jumps over the lazy dog"
tokens = tokenizer.encode(sample)
print("Text:", sample)
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

# Decode back to text
decoded = tokenizer.decode(tokens)
print("Decoded back:", decoded)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Text: The quick brown fox jumps over the lazy dog
Tokens: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290]
Number of tokens: 9
Decoded back: The quick brown fox jumps over the lazy dog


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

CONTEXT_LENGTH = 512

# Tokenize the entire dataset
def tokenize_dataset(dataset, tokenizer):
    all_tokens = []
    for example in dataset['train']:
        text = example['text'].strip()
        if len(text) > 0:
            tokens = tokenizer.encode(text)
            all_tokens.extend(tokens)
    return all_tokens

print("Tokenizing... this takes ~30 seconds")
all_tokens = tokenize_dataset(dataset, tokenizer)
print(f"Total tokens: {len(all_tokens):,}")

# Chop into chunks of 512
class TextDataset(Dataset):
    def __init__(self, tokens, context_length):
        self.examples = []
        for i in range(0, len(tokens) - context_length, context_length):
            chunk = tokens[i : i + context_length + 1]
            self.examples.append(torch.tensor(chunk, dtype=torch.long))
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        x = self.examples[idx][:-1]   # input
        y = self.examples[idx][1:]    # target (shifted by 1)
        return x, y

train_dataset = TextDataset(all_tokens, CONTEXT_LENGTH)
print(f"Number of training chunks: {len(train_dataset)}")
print(f"Each chunk shape: {train_dataset[0][0].shape}")

Tokenizing... this takes ~30 seconds
Total tokens: 2,347,038
Number of training chunks: 4584
Each chunk shape: torch.Size([512])


In [6]:
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Split into heads
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        # Causal mask (can't look at future tokens)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        # Merge heads back
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        
        x = self.token_emb(x) + self.pos_emb(positions)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_final(x)
        return self.head(x)

# Create model
VOCAB_SIZE = tokenizer.vocab_size
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 4
FF_DIM = 1024

model = Transformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Model is on: {next(model.parameters()).device}")

Model parameters: 29,022,208
Model is on: cuda:0


In [7]:
from torch.utils.data import DataLoader
import time

# Validation dataset
def tokenize_split(split):
    all_tokens = []
    for example in dataset[split]:
        text = example['text'].strip()
        if len(text) > 0:
            tokens = tokenizer.encode(text)
            all_tokens.extend(tokens)
    return all_tokens

val_tokens = tokenize_split('validation')
val_dataset = TextDataset(val_tokens, CONTEXT_LENGTH)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, val_loader):
    model.eval()
    total_loss = 0
    total_batches = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.cuda(), y.cuda()
            logits = model(x)
            loss = loss_fn(logits.view(-1, VOCAB_SIZE), y.view(-1))
            total_loss += loss.item()
            total_batches += 1
    avg_loss = total_loss / total_batches
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

def train(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        start_time = time.time()
        tokens_processed = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.cuda(), y.cuda()
            
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits.view(-1, VOCAB_SIZE), y.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            tokens_processed += x.numel()

            if batch_idx % 50 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        # Epoch summary
        epoch_time = time.time() - start_time
        throughput = tokens_processed / epoch_time
        val_loss, val_ppl = evaluate(model, val_loader)
        train_loss = total_loss / len(train_loader)

        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1} complete")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Val Perplexity: {val_ppl:.2f}")
        print(f"Throughput: {throughput:,.0f} tokens/sec")
        print(f"Time: {epoch_time:.1f}s")
        print(f"{'='*60}\n")

print("Starting training...")
train(model, train_loader, val_loader, epochs=3)

Starting training...
Epoch 1 | Batch 0/287 | Loss: 11.0034
Epoch 1 | Batch 50/287 | Loss: 7.6051
Epoch 1 | Batch 100/287 | Loss: 7.3183
Epoch 1 | Batch 150/287 | Loss: 7.1355
Epoch 1 | Batch 200/287 | Loss: 7.0248
Epoch 1 | Batch 250/287 | Loss: 6.8390

Epoch 1 complete
Train Loss: 7.3803
Val Loss:   6.8407
Val Perplexity: 935.15
Throughput: 23,471 tokens/sec
Time: 100.0s

Epoch 2 | Batch 0/287 | Loss: 6.7653
Epoch 2 | Batch 50/287 | Loss: 6.5794
Epoch 2 | Batch 100/287 | Loss: 6.5203
Epoch 2 | Batch 150/287 | Loss: 6.5321
Epoch 2 | Batch 200/287 | Loss: 6.4632
Epoch 2 | Batch 250/287 | Loss: 6.4956

Epoch 2 complete
Train Loss: 6.5872
Val Loss:   6.5400
Val Perplexity: 692.28
Throughput: 20,557 tokens/sec
Time: 114.2s

Epoch 3 | Batch 0/287 | Loss: 6.3292
Epoch 3 | Batch 50/287 | Loss: 6.3596
Epoch 3 | Batch 100/287 | Loss: 6.4137
Epoch 3 | Batch 150/287 | Loss: 6.2381
Epoch 3 | Batch 200/287 | Loss: 6.2869
Epoch 3 | Batch 250/287 | Loss: 6.1565

Epoch 3 complete
Train Loss: 6.2325
Va

In [8]:
torch.save(model.state_dict(), 'baseline_transformer.pt')
print("Model saved!")

Model saved!


In [9]:
class MultiQueryAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, self.head_dim) 
        self.v = nn.Linear(embed_dim, self.head_dim)
        
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        
        # Q split into multiple heads as before
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # K and V are shared — no head splitting needed
        # shape will be (B, T, head_dim) then unsqueeze to (B, 1, T, head_dim)
        k = self.k(x).unsqueeze(1)
        v = self.v(x).unsqueeze(1)
        
        # Attention scores — same formula as before
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        # Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [10]:
# Test MQA works
test_input = torch.randn(2, 10, 256).cuda()
mqa = MultiQueryAttention(embed_dim=256, num_heads=8).cuda()
output = mqa(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [11]:
# Swap MQA into the Transformer
class MQATransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = MultiQueryAttention(embed_dim, num_heads) 
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [12]:
class MQATransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            MQATransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)

In [13]:
mqa_model = MQATransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in mqa_model.parameters())
print(f"MQA Model parameters: {total_params:,}")

MQA Model parameters: 28,561,664


In [14]:
optimizer_mqa = torch.optim.AdamW(mqa_model.parameters(), lr=3e-4)

print("Training MQA Transformer...")
train(mqa_model, train_loader, val_loader, epochs=3)

Training MQA Transformer...
Epoch 1 | Batch 0/287 | Loss: 11.0106
Epoch 1 | Batch 50/287 | Loss: 11.0030
Epoch 1 | Batch 100/287 | Loss: 11.0128
Epoch 1 | Batch 150/287 | Loss: 11.0150
Epoch 1 | Batch 200/287 | Loss: 11.0117
Epoch 1 | Batch 250/287 | Loss: 10.9944

Epoch 1 complete
Train Loss: 11.0050
Val Loss:   11.0098
Val Perplexity: 60464.26
Throughput: 21,133 tokens/sec
Time: 111.1s

Epoch 2 | Batch 0/287 | Loss: 11.0110
Epoch 2 | Batch 50/287 | Loss: 11.0086
Epoch 2 | Batch 100/287 | Loss: 11.0115
Epoch 2 | Batch 150/287 | Loss: 11.0071
Epoch 2 | Batch 200/287 | Loss: 10.9980
Epoch 2 | Batch 250/287 | Loss: 10.9985

Epoch 2 complete
Train Loss: 11.0050
Val Loss:   11.0098
Val Perplexity: 60464.26
Throughput: 21,230 tokens/sec
Time: 110.6s

Epoch 3 | Batch 0/287 | Loss: 11.0073
Epoch 3 | Batch 50/287 | Loss: 11.0176
Epoch 3 | Batch 100/287 | Loss: 11.0041
Epoch 3 | Batch 150/287 | Loss: 11.0006
Epoch 3 | Batch 200/287 | Loss: 11.0062
Epoch 3 | Batch 250/287 | Loss: 10.9994

Epoch 

In [15]:
test_input = torch.randn(2, 10, 256).cuda()
mqa = MultiQueryAttention(embed_dim=256, num_heads=8).cuda()
out = mqa(test_input)
print(out.mean())
print(out.std())

tensor(0.0041, device='cuda:0', grad_fn=<MeanBackward0>)
tensor(0.1885, device='cuda:0', grad_fn=<StdBackward0>)


In [16]:
mqa_model = MQATransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

optimizer_mqa = torch.optim.AdamW(mqa_model.parameters(), lr=3e-4)

print("Training MQA Transformer...")
train(mqa_model, train_loader, val_loader, epochs=3)

Training MQA Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9722
Epoch 1 | Batch 50/287 | Loss: 10.9823
Epoch 1 | Batch 100/287 | Loss: 10.9778
Epoch 1 | Batch 150/287 | Loss: 10.9828
Epoch 1 | Batch 200/287 | Loss: 10.9911
Epoch 1 | Batch 250/287 | Loss: 10.9932

Epoch 1 complete
Train Loss: 10.9838
Val Loss:   10.9854
Val Perplexity: 59008.49
Throughput: 21,207 tokens/sec
Time: 110.7s

Epoch 2 | Batch 0/287 | Loss: 10.9842
Epoch 2 | Batch 50/287 | Loss: 10.9889
Epoch 2 | Batch 100/287 | Loss: 10.9844
Epoch 2 | Batch 150/287 | Loss: 10.9846
Epoch 2 | Batch 200/287 | Loss: 10.9823
Epoch 2 | Batch 250/287 | Loss: 10.9771

Epoch 2 complete
Train Loss: 10.9838
Val Loss:   10.9854
Val Perplexity: 59008.49
Throughput: 21,174 tokens/sec
Time: 110.8s

Epoch 3 | Batch 0/287 | Loss: 10.9807
Epoch 3 | Batch 50/287 | Loss: 10.9816
Epoch 3 | Batch 100/287 | Loss: 10.9746
Epoch 3 | Batch 150/287 | Loss: 10.9742
Epoch 3 | Batch 200/287 | Loss: 10.9870
Epoch 3 | Batch 250/287 | Loss: 10.9942

Epoch 

In [17]:
optimizer = optimizer_mqa
print("Training MQA Transformer...")
train(mqa_model, train_loader, val_loader, epochs=3)

Training MQA Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9843
Epoch 1 | Batch 50/287 | Loss: 7.5749
Epoch 1 | Batch 100/287 | Loss: 7.2489
Epoch 1 | Batch 150/287 | Loss: 7.1422
Epoch 1 | Batch 200/287 | Loss: 6.9302
Epoch 1 | Batch 250/287 | Loss: 6.8802

Epoch 1 complete
Train Loss: 7.3802
Val Loss:   6.8428
Val Perplexity: 937.07
Throughput: 20,891 tokens/sec
Time: 112.3s

Epoch 2 | Batch 0/287 | Loss: 6.7779
Epoch 2 | Batch 50/287 | Loss: 6.7229
Epoch 2 | Batch 100/287 | Loss: 6.5339
Epoch 2 | Batch 150/287 | Loss: 6.6625
Epoch 2 | Batch 200/287 | Loss: 6.5547
Epoch 2 | Batch 250/287 | Loss: 6.4192

Epoch 2 complete
Train Loss: 6.5940
Val Loss:   6.5527
Val Perplexity: 701.12
Throughput: 20,757 tokens/sec
Time: 113.1s

Epoch 3 | Batch 0/287 | Loss: 6.2946
Epoch 3 | Batch 50/287 | Loss: 6.2788
Epoch 3 | Batch 100/287 | Loss: 6.2157
Epoch 3 | Batch 150/287 | Loss: 6.2055
Epoch 3 | Batch 200/287 | Loss: 6.2705
Epoch 3 | Batch 250/287 | Loss: 6.2055

Epoch 3 complete
Train Loss: 6.

In [18]:
import os
os.listdir('/kaggle/working/')

['baseline_transformer.pt', '__notebook__.ipynb']

In [19]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, window_size=64):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.window_size = window_size
        
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        # Causal mask (same as before)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        
        # Window mask — block tokens further than window_size away
        # i - j > window_size means token j is too far from token i
        window_mask = torch.ones(T, T, device=x.device)
        window_mask = torch.tril(window_mask, diagonal=0)
        window_mask = torch.triu(window_mask, diagonal=self.window_size)  # what goes here?
        window_mask = (window_mask == 0)
        
        # Combine both masks
        mask = causal_mask | window_mask
        scores = scores.masked_fill(mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [20]:
test_input = torch.randn(2, 10, 256).cuda()
swa = SlidingWindowAttention(embed_dim=256, num_heads=8, window_size=64).cuda()
output = swa(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")


Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [21]:
class SWATransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = SlidingWindowAttention(embed_dim, num_heads) 
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [22]:
class SWATransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            SWATransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)

In [23]:
swa_model = SWATransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in swa_model.parameters())
print(f"SWA Model parameters: {total_params:,}")

SWA Model parameters: 29,022,208


In [24]:
optimizer_swa = torch.optim.AdamW(swa_model.parameters(), lr=3e-4)
optimizer = optimizer_swa
print("Training SWA Transformer...")
train(swa_model, train_loader, val_loader, epochs=3)

Training SWA Transformer...
Epoch 1 | Batch 0/287 | Loss: nan
Epoch 1 | Batch 50/287 | Loss: nan
Epoch 1 | Batch 100/287 | Loss: nan
Epoch 1 | Batch 150/287 | Loss: nan
Epoch 1 | Batch 200/287 | Loss: nan
Epoch 1 | Batch 250/287 | Loss: nan

Epoch 1 complete
Train Loss: nan
Val Loss:   nan
Val Perplexity: nan
Throughput: 25,237 tokens/sec
Time: 93.0s

Epoch 2 | Batch 0/287 | Loss: nan
Epoch 2 | Batch 50/287 | Loss: nan
Epoch 2 | Batch 100/287 | Loss: nan
Epoch 2 | Batch 150/287 | Loss: nan
Epoch 2 | Batch 200/287 | Loss: nan
Epoch 2 | Batch 250/287 | Loss: nan

Epoch 2 complete
Train Loss: nan
Val Loss:   nan
Val Perplexity: nan
Throughput: 25,232 tokens/sec
Time: 93.0s

Epoch 3 | Batch 0/287 | Loss: nan
Epoch 3 | Batch 50/287 | Loss: nan
Epoch 3 | Batch 100/287 | Loss: nan
Epoch 3 | Batch 150/287 | Loss: nan
Epoch 3 | Batch 200/287 | Loss: nan
Epoch 3 | Batch 250/287 | Loss: nan

Epoch 3 complete
Train Loss: nan
Val Loss:   nan
Val Perplexity: nan
Throughput: 25,233 tokens/sec
Time: 9

In [25]:
class LinearAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply φ — ELU + 1 keeps values positive
        q = torch.nn.functional.elu(q) + 1
        k = torch.nn.functional.elu(k) + 1
        
        # Linear attention trick — compute (K^T * V) first
        # k shape: (B, heads, T, head_dim)
        # v shape: (B, heads, T, head_dim)
        kv = torch.matmul(k.transpose(-2, -1), v)  # (B, heads, head_dim, head_dim)
        
        # Then multiply Q by KV
        out = torch.matmul(q, kv)  
        
        # Normalize
        k_sum = k.sum(dim=-2, keepdim=True)
        denom = (q * k_sum).sum(dim=-1, keepdim=True).clamp(min=1e-6)
        out = out / denom
        
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [26]:
test_input = torch.randn(2, 10, 256).cuda()
la = LinearAttention(embed_dim=256, num_heads=8).cuda()
output = la(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [27]:
class LinearAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = LinearAttention(embed_dim, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class LinearAttentionTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            LinearAttentionBlock(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)

In [28]:
la_model = LinearAttentionTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in la_model.parameters())
print(f"LA Model parameters: {total_params:,}")

LA Model parameters: 29,022,208


In [29]:
optimizer_la = torch.optim.AdamW(la_model.parameters(), lr=3e-4)
optimizer = optimizer_la
print("Training LA Transformer...")
train(la_model, train_loader, val_loader, epochs=3)

Training LA Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9806
Epoch 1 | Batch 50/287 | Loss: 7.5747
Epoch 1 | Batch 100/287 | Loss: 7.3321
Epoch 1 | Batch 150/287 | Loss: 7.1660
Epoch 1 | Batch 200/287 | Loss: 7.1003
Epoch 1 | Batch 250/287 | Loss: 6.8045

Epoch 1 complete
Train Loss: 7.3849
Val Loss:   6.8454
Val Perplexity: 939.54
Throughput: 23,940 tokens/sec
Time: 98.0s

Epoch 2 | Batch 0/287 | Loss: 6.7768
Epoch 2 | Batch 50/287 | Loss: 6.7224
Epoch 2 | Batch 100/287 | Loss: 6.6730
Epoch 2 | Batch 150/287 | Loss: 6.5645
Epoch 2 | Batch 200/287 | Loss: 6.4787
Epoch 2 | Batch 250/287 | Loss: 6.3118

Epoch 2 complete
Train Loss: 6.5817
Val Loss:   6.5261
Val Perplexity: 682.71
Throughput: 23,954 tokens/sec
Time: 98.0s

Epoch 3 | Batch 0/287 | Loss: 6.4472
Epoch 3 | Batch 50/287 | Loss: 6.3550
Epoch 3 | Batch 100/287 | Loss: 6.1931
Epoch 3 | Batch 150/287 | Loss: 6.0551
Epoch 3 | Batch 200/287 | Loss: 6.0877
Epoch 3 | Batch 250/287 | Loss: 6.1262

Epoch 3 complete
Train Loss: 6.205

In [30]:
class ALiBiAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
        
        # ALiBi slopes — one per head
        slopes = torch.tensor([2**(-8*i/num_heads) for i in range(1, num_heads+1)])
        self.register_buffer('slopes', slopes)
    
    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        # ALiBi bias — penalty based on distance between tokens
        positions = torch.arange(T, device=x.device)
        distances = positions.unsqueeze(0) - positions.unsqueeze(1)  # (T, T)
        distances = distances.abs()
        alibi_bias = -self.slopes.view(self.num_heads, 1, 1) * distances
        scores = scores + alibi_bias
        
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [31]:
class ALiBiTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            ALiBiTransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)

In [32]:
class ALiBiTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = ALiBiAttention(embed_dim, num_heads) 
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [33]:
alibi_model = ALiBiTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in alibi_model.parameters())
    
print(f"ALIBI Model parameters: {total_params:,}")

ALIBI Model parameters: 29,022,208


In [34]:
optimizer_alibi = torch.optim.AdamW(alibi_model.parameters(), lr=3e-4)
optimizer = optimizer_alibi
print("Training ALIBI Transformer...")
train(alibi_model, train_loader, val_loader, epochs=3)

Training ALIBI Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9785
Epoch 1 | Batch 50/287 | Loss: 7.6138
Epoch 1 | Batch 100/287 | Loss: 7.2056
Epoch 1 | Batch 150/287 | Loss: 6.9788
Epoch 1 | Batch 200/287 | Loss: 6.9955
Epoch 1 | Batch 250/287 | Loss: 6.8834

Epoch 1 complete
Train Loss: 7.3250
Val Loss:   6.7328
Val Perplexity: 839.45
Throughput: 20,068 tokens/sec
Time: 117.0s

Epoch 2 | Batch 0/287 | Loss: 6.5466
Epoch 2 | Batch 50/287 | Loss: 6.5457
Epoch 2 | Batch 100/287 | Loss: 6.5969
Epoch 2 | Batch 150/287 | Loss: 6.6225
Epoch 2 | Batch 200/287 | Loss: 6.3739
Epoch 2 | Batch 250/287 | Loss: 6.2304

Epoch 2 complete
Train Loss: 6.4415
Val Loss:   6.3797
Val Perplexity: 589.73
Throughput: 19,988 tokens/sec
Time: 117.4s

Epoch 3 | Batch 0/287 | Loss: 6.1684
Epoch 3 | Batch 50/287 | Loss: 6.1846
Epoch 3 | Batch 100/287 | Loss: 6.1013
Epoch 3 | Batch 150/287 | Loss: 6.0930
Epoch 3 | Batch 200/287 | Loss: 6.0076
Epoch 3 | Batch 250/287 | Loss: 5.8843

Epoch 3 complete
Train Loss: 

In [35]:
class RoPEAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def get_rotary_matrix(self, T, device):
        theta = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim))
        positions = torch.arange(T, device=device).float()
        angles = torch.outer(positions, theta)
        cos = torch.cos(angles)
        sin = torch.sin(angles)
        return cos, sin
    
    def rotate(self, x, cos, sin):
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        rotated = torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1)
        return rotated.flatten(-2)
    
    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        cos, sin = self.get_rotary_matrix(T, x.device)
        q = self.rotate(q, cos, sin)
        k = self.rotate(k, cos, sin)
        
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [36]:
class RoPETransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            RoPETransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)

In [37]:
class RoPETransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = RoPEAttention(embed_dim, num_heads) 
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [38]:
rope_model = RoPETransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in rope_model.parameters())
    
print(f"ROPE Model parameters: {total_params:,}")

ROPE Model parameters: 29,022,208


In [39]:
optimizer_rope = torch.optim.AdamW(rope_model.parameters(), lr=3e-4)
optimizer = optimizer_rope
print("Training ROPE Transformer...")
train(rope_model, train_loader, val_loader, epochs=3)

Training ROPE Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9896
Epoch 1 | Batch 50/287 | Loss: 7.7010
Epoch 1 | Batch 100/287 | Loss: 7.2913
Epoch 1 | Batch 150/287 | Loss: 7.0154
Epoch 1 | Batch 200/287 | Loss: 6.8390
Epoch 1 | Batch 250/287 | Loss: 6.7530

Epoch 1 complete
Train Loss: 7.3553
Val Loss:   6.7518
Val Perplexity: 855.62
Throughput: 20,009 tokens/sec
Time: 117.3s

Epoch 2 | Batch 0/287 | Loss: 6.4479
Epoch 2 | Batch 50/287 | Loss: 6.6282
Epoch 2 | Batch 100/287 | Loss: 6.6466
Epoch 2 | Batch 150/287 | Loss: 6.4721
Epoch 2 | Batch 200/287 | Loss: 6.2684
Epoch 2 | Batch 250/287 | Loss: 6.2417

Epoch 2 complete
Train Loss: 6.4536
Val Loss:   6.3841
Val Perplexity: 592.33
Throughput: 20,001 tokens/sec
Time: 117.3s

Epoch 3 | Batch 0/287 | Loss: 6.2720
Epoch 3 | Batch 50/287 | Loss: 6.2169
Epoch 3 | Batch 100/287 | Loss: 6.1292
Epoch 3 | Batch 150/287 | Loss: 6.1659
Epoch 3 | Batch 200/287 | Loss: 5.9723
Epoch 3 | Batch 250/287 | Loss: 5.9331

Epoch 3 complete
Train Loss: 6

In [40]:
class ConvAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, kernel_size=3):
        super().__init__()
        # Conv1D to capture local patterns before attention
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=embed_dim,
            kernel_size=kernel_size,
            padding=kernel_size//2,
            groups=embed_dim  # depthwise convolution
        )
        self.attn = RoPEAttention(embed_dim, num_heads)  # best attention from Q2
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ln3 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        # Conv expects (B, C, T) but x is (B, T, C) — transpose first
        x = x + self.conv(self.ln1(x).transpose(1,2)).transpose(1,2)
        x = x + self.attn(self.ln2(x))
        x = x + self.ff(self.ln3(x))
        return x

In [41]:
class ConvTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length, ff_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.ModuleList([
            RoPETransformerBlock(embed_dim, num_heads, ff_dim) 
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.head(x)
class ConvTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = RoPEAttention(embed_dim, num_heads) 
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [42]:
conv_model = ConvTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    context_length=CONTEXT_LENGTH,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS
).cuda()

total_params = sum(p.numel() for p in conv_model.parameters())
    
print(f"CONV Model parameters: {total_params:,}")

CONV Model parameters: 29,022,208


In [43]:
optimizer_conv = torch.optim.AdamW(conv_model.parameters(), lr=3e-4)
optimizer = optimizer_conv
print("Training CONV Transformer...")
train(conv_model, train_loader, val_loader, epochs=3)

Training CONV Transformer...
Epoch 1 | Batch 0/287 | Loss: 10.9883
Epoch 1 | Batch 50/287 | Loss: 7.5590
Epoch 1 | Batch 100/287 | Loss: 7.3202
Epoch 1 | Batch 150/287 | Loss: 7.0097
Epoch 1 | Batch 200/287 | Loss: 6.8576
Epoch 1 | Batch 250/287 | Loss: 6.7960

Epoch 1 complete
Train Loss: 7.3454
Val Loss:   6.7524
Val Perplexity: 856.12
Throughput: 20,025 tokens/sec
Time: 117.2s

Epoch 2 | Batch 0/287 | Loss: 6.7208
Epoch 2 | Batch 50/287 | Loss: 6.5629
Epoch 2 | Batch 100/287 | Loss: 6.5659
Epoch 2 | Batch 150/287 | Loss: 6.4838
Epoch 2 | Batch 200/287 | Loss: 6.2186
Epoch 2 | Batch 250/287 | Loss: 6.3147

Epoch 2 complete
Train Loss: 6.4581
Val Loss:   6.3956
Val Perplexity: 599.21
Throughput: 19,896 tokens/sec
Time: 118.0s

Epoch 3 | Batch 0/287 | Loss: 6.0057
Epoch 3 | Batch 50/287 | Loss: 6.0778
Epoch 3 | Batch 100/287 | Loss: 6.0973
Epoch 3 | Batch 150/287 | Loss: 6.0766
Epoch 3 | Batch 200/287 | Loss: 6.0948
Epoch 3 | Batch 250/287 | Loss: 6.0252

Epoch 3 complete
Train Loss: 6

In [44]:
[]

[]